# 加载模型并打印模型信息

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("正在加载模型（首次运行会下载约 500MB）...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在加载模型（首次运行会下载约 500MB）...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1301.28it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

# 代码实验一：亲眼看"下一个词预测"

In [2]:
import torch

prompt = "I like"

input_ids = tokenizer.encode(prompt, return_tensors="pt")  # 文字 → token ID
tokens = [tokenizer.decode(id) for id in input_ids[0]]
logits = model(input_ids).logits[0, -1, :]   # 模型输出 50257 个原始得分
probs = torch.softmax(logits, dim=-1)          # 得分 → 概率
next_token = torch.argmax(probs)               # 选概率最高的

# 模型认为最可能的topK个词
top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)
print(f"\n模型预测 '{prompt}' 后面最可能的 {top_k} 个词：")
print("-" * 50)
for i in range(top_k):
    token = tokenizer.decode(top_indices[i])
    prob = top_probs[i].item() * 100
    bar = "█" * int(prob / 2)
    print(f"  {i+1:2d}. '{token}' \t {prob:5.1f}%  {bar}")

print(f"Token IDs: {input_ids.tolist()[0]}")
print(f"tokens: {tokens}")
print(f"Token logits: {logits.tolist()}")
print(f"Token probs: {probs.tolist()}")
print(f"next token: {next_token}")


模型预测 'I like' 后面最可能的 10 个词：
--------------------------------------------------
   1. ' to' 	  23.3%  ███████████
   2. ' the' 	  12.8%  ██████
   3. ' it' 	   9.2%  ████
   4. ' that' 	   6.1%  ███
   5. ' this' 	   4.5%  ██
   6. ' my' 	   2.7%  █
   7. ' a' 	   1.6%  
   8. ' what' 	   1.5%  
   9. ' how' 	   1.4%  
  10. ' you' 	   1.3%  
Token IDs: [40, 588]
tokens: ['I', ' like']
Token logits: [-73.52761840820312, -72.83938598632812, -79.3030014038086, -80.35306549072266, -77.95025634765625, -78.43585205078125, -74.89720153808594, -75.77147674560547, -73.66260528564453, -77.65715789794922, -78.74474334716797, -69.91780853271484, -73.17950439453125, -70.71410369873047, -73.6757583618164, -79.18724060058594, -78.38652038574219, -77.48112487792969, -77.34650421142578, -77.95329284667969, -78.65010070800781, -78.9052963256836, -78.73460388183594, -78.69767761230469, -79.3536148071289, -72.31941223144531, -73.88880157470703, -78.97332000732422, -78.02950286865234, -77.7787094116211, -

# 代码实验二：逐词生成与Temperature

In [6]:
import torch
import time

def generate_step_by_step(prompt, max_tokens=30, temperature=0.8, verbose=True):
    """
    逐个 token 生成文本，展示完整的预测过程。

    参数：
        prompt: 输入文本
        max_tokens: 最多生成多少个 token
        temperature: 控制"创造力"。越高越随机，越低越确定
        verbose: 是否打印每一步的细节
    """
    if verbose:
        print(f"输入: '{prompt}'")
        print(f"Temperature: {temperature}")
        print("=" * 60)
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    generated_text = prompt

    for step in range(max_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
            next_token_logits = outputs.logits[0, -1, :]
        # Temperature 缩放
        # temperature > 1: 概率更平均 → 输出更随机、更"有创意"
        # temperature < 1: 概率更集中 → 输出更确定、更"保守"
        # temperature = 1: 不做调整
        scaled_logits = next_token_logits / temperature
        probabilities = torch.softmax(scaled_logits, dim=-1)

        # 从概率分布中随机采样（而不是总选最高的）
        next_token_id = torch.multinomial(probabilities, 1)
        next_token = tokenizer.decode(next_token_id[0])

        # 这个 token 被选中的概率
        token_prob = probabilities[next_token_id[0]].item() * 100

        if verbose:
            # 同时展示 Top 3 候选，让你看到"没被选中的可能性"
            top3_probs, top3_indices = torch.topk(probabilities, 3)
            alternatives = [
                f"'{tokenizer.decode(top3_indices[j])}' {top3_probs[j].item()*100:.1f}%"
                for j in range(3)
            ]
            chosen_marker = " ✓" if next_token_id[0] in top3_indices else ""
            print(
                f"  Step {step+1:2d}: 选了 '{next_token}' ({token_prob:.1f}%){chosen_marker}"
                f"   | Top 3: {', '.join(alternatives)}"
            )

        # 拼接到输入，继续下一轮
        input_ids = torch.cat([input_ids, next_token_id.unsqueeze(0)], dim=1)
        generated_text += next_token

        # 模拟打字机效果
        if verbose:
            time.sleep(0.05)

        # 遇到句号就停止
        if next_token.strip() in [".", "!", "?"]:
            break

    if verbose:
        print("=" * 60)
        print(f"完整输出: '{generated_text}'")

    return generated_text

In [7]:
prompt = "Thank you very"
max_tokens = 30
temperature = 0.8

generate_step_by_step(prompt=prompt, max_tokens=max_tokens, temperature=temperature)

输入: 'Thank you very'
Temperature: 0.8
  Step  1: 选了 ' much' (99.8%) ✓   | Top 3: ' much' 99.8%, ' very' 0.1%, ',' 0.1%
  Step  2: 选了 ' for' (61.4%) ✓   | Top 3: ' for' 61.4%, ',' 16.3%, '.' 8.9%
  Step  3: 选了 ' your' (41.1%) ✓   | Top 3: ' your' 41.1%, ' the' 11.7%, ' reading' 11.3%
  Step  4: 选了 ' support' (32.9%) ✓   | Top 3: ' support' 32.9%, ' time' 12.6%, ' continued' 8.3%
  Step  5: 选了 ',' (10.0%)   | Top 3: '.' 37.5%, ' and' 16.0%, '!' 10.6%
  Step  6: 选了 '
' (10.7%) ✓   | Top 3: ' and' 43.6%, '
' 10.7%, ' I' 8.7%
  Step  7: 选了 '
' (100.0%) ✓   | Top 3: '
' 100.0%, '"' 0.0%, '[' 0.0%
  Step  8: 选了 'B' (1.7%)   | Top 3: 'J' 2.8%, 'Mike' 2.8%, 'Michael' 2.3%
  Step  9: 选了 'ryan' (13.0%) ✓   | Top 3: 'ryan' 13.0%, 'obby' 8.4%, 'eth' 6.4%
  Step 10: 选了 '.' (27.3%) ✓   | Top 3: '
' 37.5%, '.' 27.3%, '<|endoftext|>' 5.3%
完整输出: 'Thank you very much for your support,

Bryan.'


'Thank you very much for your support,\n\nBryan.'